In [ ]:
import threading
import time
a,b = threading.Lock(),threading.Lock()
def dead1():
    name = threading.current_thread().name
    print(f"[{name}]尝试获取a")
    with a:
        print(f"[{name}]已获取a")
        time.sleep(1)
        print(f"[{name}] 尝试获取 b")
        with b: 
            print(f"[{name}] 已获取 b,dead1 完成")
def dead2():
    name = threading.current_thread().name
    print(f"[{name}] 尝试获取 b")
    with b:
        print(f"[{name}] 已获取 b")
        time.sleep(1)
        print(f"[{name}] 尝试获取 a")
        with a: 
            print(f"[{name}] 已获取 a,dead2 完成")
if __name__ == "__main__":
    t1 = threading.Thread(target=dead1,name="T1")
    t2 = threading.Thread(target=dead2,name="T2")
    t1.daemon = True
    t2.daemon = True
    t1.start()
    t2.start()
    t1.join(timeout=1)
    t2.join(timeout=1)
    print("T1 is_alive:",t1.is_alive())
    print("T2 is_alive:",t2.is_alive())
    if t1.is_alive() and t2.is_alive():
        print("检测到死锁：T1 持有 a 等 b,T2 持有 b 等 a")

[T1]尝试获取a
[T1]已获取a
[T2] 尝试获取 b
[T2] 已获取 b
[T2] 尝试获取 a[T1] 尝试获取 b

T1 is_alive: True
T2 is_alive: True
检测到死锁：T1 持有 a 等 b，T2 持有 b 等 a


In [2]:
import threading
import time
sem = threading.Semaphore(2)
def limited(i):
    with sem:
        start = time.strftime("%H:%M:%S",time.localtime())
        print(f"[{start}]线程{i}开始")
        time.sleep(0.3)
        end = time.strftime("%H:%M:%S",time.localtime())
        print(f"[{end}]线程{i}结束")
if __name__ == "__main__":
    threads = []
    for i in range(6):
        t = threading.Thread(target=limited,args=(i,))
        threads.append(t)
        t.start()
        for t in threads:
            t.join()
        print("所有线程执行完毕")

[09:44:51]线程0开始
[09:44:51]线程0结束
所有线程执行完毕
[09:44:51]线程1开始
[09:44:51]线程1结束
所有线程执行完毕
[09:44:51]线程2开始
[09:44:51]线程2结束
所有线程执行完毕
[09:44:51]线程3开始
[09:44:52]线程3结束
所有线程执行完毕
[09:44:52]线程4开始
[09:44:52]线程4结束
所有线程执行完毕
[09:44:52]线程5开始
[09:44:52]线程5结束
所有线程执行完毕


In [4]:
import threading
import time
ready = threading.Event()
def waiter(n):
    print(f"{n}等待信号")
    ready.wait()
    print(f"{n}收到信号,开工")
if __name__ == "__main__":
    for i in range(3):
        threading.Thread(target=waiter,args=(i,),daemon=True).start()
        time.sleep(0.3)
        print("主线程发出信号!")
        ready.set()
        time.sleep(0.1)
        print("主线程结束")

0等待信号
主线程发出信号!
0收到信号,开工
主线程结束
1等待信号
1收到信号,开工
主线程发出信号!
主线程结束
2等待信号
2收到信号,开工
主线程发出信号!
主线程结束


In [5]:
import threading
import time
buf,cap = [],3
cond = threading.Condition()
def producer():
    for i in range(6):
        with cond:
            while len(buf) >= cap:
                print(f"[{time.strftime('%H:%M:%S')}]缓冲区满,生产者等待...")
                cond.wait()
            buf.append(i)
            print(f"[{time.strftime('%H:%M:%S')}]生产{i},缓冲区:{buf}")
            cond.notify_all()
        time.sleep(0.1)
def consumer():
    for _ in range(6):
        with cond:
            while not buf:
                print(f"[{time.strftime('%H:%M:%S')}]缓冲区空,消费者等待...")
                cond.wait()
            item = buf.pop(0)
            print(f"[{time.strftime('%H:%M:%S')}]消费{item},缓冲区:{buf}")
            cond.notify_all()
        time.sleep(0.2)
if __name__ == "__main__":
    t1 = threading.Thread(target=producer,name="Producer")
    t2 = threading.Thread(target=consumer,name="Consumer")
    t1.daemon = True
    t2.daemon = True
    t1.start()
    t2.start()
    time.sleep(3)
    print("主线程结束")

[10:22:00]生产0,缓冲区:[0]
[10:22:00]消费0,缓冲区:[]
[10:22:00]生产1,缓冲区:[1]
[10:22:00]生产2,缓冲区:[1, 2]
[10:22:00]消费1,缓冲区:[2]
[10:22:00]生产3,缓冲区:[2, 3]
[10:22:00]消费2,缓冲区:[3]
[10:22:00]生产4,缓冲区:[3, 4]
[10:22:00]生产5,缓冲区:[3, 4, 5]
[10:22:00]消费3,缓冲区:[4, 5]
[10:22:00]消费4,缓冲区:[5]
[10:22:01]消费5,缓冲区:[]
主线程结束


In [8]:
from queue import Queue,Empty,Full
q = Queue(maxsize=5)
def producer_q(q):
    for i in range(10):
        q.put(i)
        time.sleep(0.05)
    q.put(None)
def consumer_q(q):
    while True:
        item = q.get()
        if item is None:
            break
        print("处理",item)
        q.task_done()
threading.Thread(target=producer_q,args=(q,),daemon=True).start()
threading.Thread(target=consumer_q,args=(q,),daemon=True).start()
q.join()
print("队列处理完毕")    

处理 0
队列处理完毕


处理 1
处理 2
处理 3
处理 4
处理 5
处理 6
处理 7
处理 8
处理 9


In [16]:
import asyncio
async def hello(name):
    print(f"{name}开始")
    await asyncio.sleep(1)
    print(f"{name}结束")
    return f"{name}完成"
result = await hello("A")
print(result)

A开始
A结束
A完成


In [19]:
async def main():
    results = await asyncio.gather(hello("A"),hello("B"),hello("C"))
    print(results)
    tasks = [asyncio.create_task(hello(f"T{i}")) for i in range(5)]
    for done in asyncio.as_completed(tasks):
        print(await done)
await main()

A开始
B开始
C开始
A结束
B结束
C结束
['A完成', 'B完成', 'C完成']
T0开始
T1开始
T2开始
T3开始
T4开始
T0结束
T2结束
T4结束
T1结束
T3结束
T0完成
T2完成
T4完成
T1完成
T3完成


In [20]:
import asyncio
import time
async def slow():
    await asyncio.sleep(10);return "ok"
async def with_timeout():
    try:
        res = await asyncio.wait_for(slow(),timeout=1.0)
    except asyncio.TimeoutError:
        print("超时了")
await with_timeout()

超时了


In [21]:
class AsyncTimer:
    async def __aenter__(self):
        self._t = asyncio.get_event_loop().time();print("开始计时")
        return self
    async def __aexit__(self,exc_type,exc,tb):
        print(f"耗时{asyncio.get_event_loop().time()-self._t:.3f}s")
class AsyncRange:
    def __init__(self,n):
        self.n = n
    def __aiter__(self):
        self.i = 0
        return self
    async def __anext__(self):
        if self.i >= self.n:
            raise StopAsyncIteration
        await asyncio.sleep(0.1)
        self.i += 1
        return self.i - 1
async def demo():
    async with AsyncTimer():
        async for x in AsyncRange(3):
            print(x)
await demo()

开始计时
0
1
2
耗时0.328s
